# 第二十六课:LoRA——低秩补丁

讲义:[26-lora.md](26-lora.md)。冻结底座,只训 1.70% 的参数:低秩补丁 ΔW=(α/r)BA。含 merge 验收、遗忘分解、挂载位置与秩的对照实验。

In [ ]:
import math

import time

import numpy as np



# ===== 第 20 课 MiniGPT 原码(含 generate) =====

def softmax(x, axis=-1):

    m = np.max(x, axis=axis, keepdims=True)

    e = np.exp(x - m)

    return e / e.sum(axis=axis, keepdims=True)



class MiniGPT:

    def __init__(self, vocab=16, d=32, n_heads=4, n_layers=2, seed=0):

        rng = np.random.default_rng(seed)

        s = 0.1

        self.E = rng.normal(size=(vocab, d)) * 0.5

        self.dh = d // n_heads

        self.H, self.D, self.NL = n_heads, d, n_layers

        self.layers = [{

            'Wq': rng.normal(size=(d, d))*s, 'Wk': rng.normal(size=(d, d))*s,

            'Wv': rng.normal(size=(d, d))*s, 'Wo': rng.normal(size=(d, d))*s,

            'W1': rng.normal(size=(d, 4*d))*s*0.8, 'W2': rng.normal(size=(4*d, d))*s*0.8,

            'W3': rng.normal(size=(d, 4*d))*s*0.8,

        } for _ in range(n_layers)]

        self.head = rng.normal(size=(d, vocab)) * s



    def rms(self, x, eps=1e-5):

        return x / np.sqrt((x**2).mean(-1, keepdims=True) + eps)



    def rope(self, x, pos):

        half = x.shape[-1] // 2

        ang = pos[:, None] / (10000 ** (np.arange(half)*2/x.shape[-1]))

        c, s = np.cos(ang), np.sin(ang)

        x1, x2 = x[..., :half], x[..., half:]

        return np.concatenate([x1*c - x2*s, x1*s + x2*c], -1)



    def _attn(self, x, pos, W, cache):

        n = x.shape[0]

        q, k, v = self.rope(x@W['Wq'], pos), self.rope(x@W['Wk'], pos), x@W['Wv']

        if cache is not None:

            k, v = np.concatenate([cache[0], k]), np.concatenate([cache[1], v])

        T = k.shape[0]

        mask = np.triu(np.ones((n, T), bool), T - n + 1)

        outs = []

        for h in range(self.H):

            sl = slice(h*self.dh, (h+1)*self.dh)

            s = q[:, sl] @ k[:, sl].T / math.sqrt(self.dh)

            s = np.where(mask, -np.inf, s)

            outs.append(softmax(s) @ v[:, sl])

        return np.concatenate(outs, -1) @ W['Wo'], (k, v)



    def _ffn(self, x, W):

        h = x @ W['W1']

        return (h / (1 + np.exp(-h)) * (x @ W['W3'])) @ W['W2']



    def forward(self, ids, cache=None):

        pos0 = cache[0][0].shape[0] if cache else 0

        x = self.E[list(ids)]

        new_cache = []

        for i, W in enumerate(self.layers):

            a, kv = self._attn(x, np.arange(pos0, pos0+len(ids)), W,

                               cache[i] if cache else None)

            x = x + a

            x = x + self._ffn(self.rms(x), W)

            new_cache.append(kv)

        logits = self.rms(x) @ self.head

        assert np.isfinite(logits).all(), 'NaN! 查掩码/精度/ε'

        return logits, new_cache



    def generate(self, ids, n, T=1.0, top_p=1.0, rng=None):

        rng = rng or np.random.default_rng()

        logits, cache = self.forward(ids)

        out = []

        for _ in range(n):

            p = softmax(logits[-1] / T)

            if top_p < 1.0:

                order = np.argsort(-p); cum = np.cumsum(p[order])

                keep = order[:np.searchsorted(cum, top_p)+1]

                m = np.zeros_like(p); m[keep] = p[keep]; p = m/m.sum()

            nxt = int(rng.choice(len(p), p=p/p.sum()))

            out.append(nxt)

            logits, cache = self.forward([nxt], cache)

        return list(ids) + out

In [ ]:
# ===== 第 23 课反向 + 本课 mask 扩展(合并为一个函数;mask=None 即 23 课原版) =====

def rms_backward(dy, x, eps=1e-5):

    r = np.sqrt((x**2).mean(-1, keepdims=True) + eps)

    y = x / r

    return (dy - y * (dy * y).mean(-1, keepdims=True)) / r



def rope_backward(dy, pos, dim):

    half = dim // 2

    ang = pos[:, None] / (10000 ** (np.arange(half)*2/dim))

    c, s = np.cos(ang), np.sin(ang)

    dy1, dy2 = dy[..., :half], dy[..., half:]

    return np.concatenate([dy1*c + dy2*s, -dy1*s + dy2*c], -1)



def silu_grad(u):

    sig = 1 / (1 + np.exp(-u))

    return sig * (1 + u * (1 - sig))



def loss_and_grads(gpt, ids, targets, mask=None):

    """23 课整机反向 + 25 课可选 mask(监督位加权)。mask=None 等价 23 课原版。"""

    T = len(ids)

    if mask is None:

        mask = [1.0] * T

    marr = np.array(mask, dtype=float)

    Ns = marr.sum()

    pos = np.arange(T)

    tape = []

    x = gpt.E[list(ids)]

    for W in gpt.layers:

        q, k, v = gpt.rope(x@W['Wq'], pos), gpt.rope(x@W['Wk'], pos), x@W['Wv']

        m = np.triu(np.ones((T, T), bool), 1)

        ps, outs = [], []

        for h in range(gpt.H):

            sl = slice(h*gpt.dh, (h+1)*gpt.dh)

            s = q[:, sl] @ k[:, sl].T / math.sqrt(gpt.dh)

            s = np.where(m, -np.inf, s)

            p = softmax(s)

            ps.append(p); outs.append(p @ v[:, sl])

        co = np.concatenate(outs, -1)

        xm = x + co @ W['Wo']

        xr = gpt.rms(xm)

        u, zv = xr @ W['W1'], xr @ W['W3']

        gate = u/(1+np.exp(-u))*zv

        tape.append(dict(x_in=x, xm=xm, xr=xr, u=u, zv=zv, gate=gate, co=co, q=q, k=k, v=v, ps=ps))

        x = xm + gate @ W['W2']                 # 注意:x 入 tape 在覆盖之前(25 课战争故事②)



    xf = x

    logits = gpt.rms(xf) @ gpt.head

    z = logits - logits.max(-1, keepdims=True)

    logp = z - np.log(np.exp(z).sum(-1, keepdims=True))

    loss = -(logp[np.arange(T), np.array(targets)] * marr).sum() / Ns



    grads = {'E': np.zeros_like(gpt.E), 'head': np.zeros_like(gpt.head),

             'layers': [dict.fromkeys(W) for W in gpt.layers]}

    p = softmax(logits)

    onehot = np.zeros_like(p)

    onehot[np.arange(T), np.array(targets)] = 1.0

    dlogits = (p - onehot) * (marr / Ns)[:, None]      # ← mask 版与 23 课的核心差别

    grads['head'] = gpt.rms(xf).T @ dlogits

    dx = rms_backward(dlogits @ gpt.head.T, xf)

    for i in reversed(range(gpt.NL)):

        W, t = gpt.layers[i], tape[i]

        dg = dx @ W['W2'].T

        du = dg * silu_grad(t['u']) * t['zv']

        dzv = dg * (t['u']/(1+np.exp(-t['u'])))

        grads['layers'][i]['W2'] = t['gate'].T @ dx

        grads['layers'][i]['W1'] = t['xr'].T @ du

        grads['layers'][i]['W3'] = t['xr'].T @ dzv

        dxm = dx + rms_backward(du @ W['W1'].T + dzv @ W['W3'].T, t['xm'])

        dc = dxm @ W['Wo'].T

        grads['layers'][i]['Wo'] = t['co'].T @ dxm

        dvs, dqs, dks = [], [], []

        for h in range(gpt.H):

            sl = slice(h*gpt.dh, (h+1)*gpt.dh)

            p_h, v_h, q_h, k_h = t['ps'][h], t['v'][:, sl], t['q'][:, sl], t['k'][:, sl]

            dO = dc[:, sl]

            dvs.append(p_h.T @ dO)

            dp = dO @ v_h.T

            ds = p_h * (dp - (dp*p_h).sum(-1, keepdims=True))

            dqs.append(ds @ k_h / math.sqrt(gpt.dh))

            dks.append(ds.T @ q_h / math.sqrt(gpt.dh))

        dv = np.concatenate(dvs, -1)

        dq = rope_backward(np.concatenate(dqs, -1), pos, gpt.D)

        dk = rope_backward(np.concatenate(dks, -1), pos, gpt.D)

        grads['layers'][i]['Wv'] = t['x_in'].T @ dv

        grads['layers'][i]['Wq'] = t['x_in'].T @ dq

        grads['layers'][i]['Wk'] = t['x_in'].T @ dk

        dx = dxm + dq @ W['Wq'].T + dk @ W['Wk'].T + dv @ W['Wv'].T

    np.add.at(grads['E'], np.array(ids), dx)

    return loss, grads



def flat_params(gpt):

    ps = [gpt.E, gpt.head]

    for W in gpt.layers:

        ps += [W[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]

    return ps



def flat_grads(grads):

    gs = [grads['E'], grads['head']]

    for L in grads['layers']:

        gs += [L[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]

    return gs



def clip_norms(gs, max_norm=1.0):

    total = math.sqrt(sum((g ** 2).sum() for g in gs))

    if total > max_norm:

        s = max_norm / total

        for g in gs:

            g *= s

    return total



class MultiAdamW:

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, wd=0.01):

        self.ps, self.lr, self.b1, self.b2, self.eps, self.wd = params, lr, *betas, eps, wd

        self.m = [np.zeros_like(p) for p in params]

        self.v = [np.zeros_like(p) for p in params]

        self.t = 0

    def step(self, gs):

        self.t += 1

        for p, g, m, v in zip(self.ps, gs, self.m, self.v):

            m *= self.b1; m += (1 - self.b1) * g

            v *= self.b2; v += (1 - self.b2) * g * g

            p -= self.lr * (m/(1-self.b1**self.t)/(np.sqrt(v/(1-self.b2**self.t))+self.eps) + self.wd*p)

def lr_at(step, total, base_lr=1e-3, warmup=100, floor_ratio=0.1):
    if step < warmup:
        return base_lr * (step + 1) / warmup
    p = (step - warmup) / max(1, total - warmup)
    return base_lr * (floor_ratio + (1 - floor_ratio) * 0.5 * (1 + math.cos(math.pi * p)))

In [ ]:
# ===== 数据 + 模板 + 任务(讲义 §3、§5) =====

text = open("data/xiyouji.txt", encoding="utf-8").read()

chars = sorted(set(text))

stoi = {c: i for i, c in enumerate(chars)}

itos = {i: c for c, i in stoi.items()}

V0 = len(chars)

data = np.array([stoi[c] for c in text], dtype=np.int32)

n_train = int(len(data) * 0.99)

train_data, val_data = data[:n_train], data[n_train:]



T_USER, T_ASSI, T_END = V0, V0 + 1, V0 + 2



def enc(s):

    return [stoi[c] for c in s]



def extend_vocab(m, seed=1):

    r = np.random.default_rng(seed)

    m.E = np.vstack([m.E, r.normal(size=(3, m.D)) * 0.1])

    m.head = np.hstack([m.head, r.normal(size=(m.D, 3)) * 0.1])



ANTI = [("大","小"),("上","下"),("天","地"),("古","今"),("来","去"),("内","外"),

        ("高","低"),("长","短"),("黑","白"),("好","坏"),("真","假"),("生","死"),

        ("冷","热"),("快","慢"),("强","弱"),("轻","重"),("多","少"),("有","无"),

        ("明","暗"),("前","后"),("左","右"),("日","夜"),("开","关"),("爱","恨"),

        ("笑","哭"),("进","退"),("买","卖"),("胜","败"),("深","浅"),("忙","闲")]



COPY_STRINGS = ["金箍", "行者", "云端", "宝殿", "妖魔", "蒲团", "袈裟", "铁棒",

                "猴王", "灵山", "仙丹", "蟠桃", "佛祖", "天河", "石猴", "宝象",

                "法术", "神通", "筋斗", "火眼"]



def make_anti_ex(pair):

    q, a = pair

    ids = [T_USER] + enc(f"反义词:{q}") + [T_ASSI] + enc(a) + [T_END]

    mask = [0] * len(ids); mask[-2] = 1; mask[-1] = 1

    return ids, mask



def make_copy_ex(s):

    ids = [T_USER] + enc(f"重复:{s}") + [T_ASSI] + enc(s) + [T_END]

    mask = [0] * len(ids)

    for j in range(-len(s)-1, 0):

        mask[j] = 1

    return ids, mask



ALL_TASKS = [make_anti_ex(p) for p in ANTI] + [make_copy_ex(s) for s in COPY_STRINGS]

SPECIAL = {T_USER: '<|user|>', T_ASSI: '<|assistant|>', T_END: '<|end|>'}

def render(ids):

    return ''.join(SPECIAL[i] if i in SPECIAL else itos[i] for i in ids)

print(f"任务样本:{len(ANTI)} 反义词 + {len(COPY_STRINGS)} 复制 = {len(ALL_TASKS)}")

print("示例:", render(ALL_TASKS[0][0]), "| mask:", ALL_TASKS[0][1])

def val_loss_no_special(m):
    """全词表 val loss 与『去掉 3 个特殊 token 后在原词表上重归一化』的 val loss。"""
    rv = np.random.default_rng(7)
    off = rv.integers(0, len(val_data) - 65, 16)
    tot_full = tot_no = 0.0
    for o in off:
        xb = list(val_data[o:o+65])
        logits, _ = m.forward(xb[:-1])
        tgt = np.array(xb[1:])
        z = logits - logits.max(-1, keepdims=True)
        logp = z - np.log(np.exp(z).sum(-1, keepdims=True))
        tot_full += -logp[np.arange(len(tgt)), tgt].mean()
        z2 = logits[:, :V0] - logits[:, :V0].max(-1, keepdims=True)
        logp2 = z2 - np.log(np.exp(z2).sum(-1, keepdims=True))
        tot_no += -logp2[np.arange(len(tgt)), tgt].mean()
    return tot_full / 16, tot_no / 16

In [ ]:
# ===== LoRA 四件套(讲义 §3) =====
LORA_ALL = ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W3', 'W2')      # 挂全部线性层(§7 有为什么)
LORA_ATTN = ('Wq', 'Wk', 'Wv', 'Wo')                        # 仅 attention(对照组)

def init_lora(gpt, keys=LORA_ALL, r=4, alpha=8, seed=1):
    """A ~ N(0,0.02),B = 0 → 初始 ΔW = 0。同时快照 W_base。"""
    rng = np.random.default_rng(seed)
    adpts, W_base = {}, {}
    for i in range(gpt.NL):
        for k in keys:
            W = gpt.layers[i][k]
            out_, in_ = W.shape
            adpts[(i, k)] = (rng.normal(size=(r, in_)) * 0.02, np.zeros((out_, r)))
            W_base[(i, k)] = W.copy()
    return adpts, W_base, alpha / r

def apply_lora(gpt, adpts, scale, W_base):
    """每步从 W_base 快照干净重建合并权重——绝不在旧账上做加减(§8 战争故事①)。"""
    for key in W_base:
        i, k = key
        A, B = adpts[key]
        gpt.layers[i][k] = W_base[key] + scale * (B @ A)

def lora_grads(grads, adpts, scale):
    """gW(合并权重梯度)投影:gA = scale·BᵀgW,gB = scale·gW·Aᵀ。
    迭代序与 flat_adapters 一致(sorted)——zip 配对不许错位(§8 战争故事②)。"""
    out = []
    for key in sorted(adpts):
        i, k = key
        A, B = adpts[key]
        gW = grads['layers'][i][k]
        out.append(scale * (B.T @ gW))
        out.append(scale * (gW @ A.T))
    return out

def flat_adapters(adpts):
    out = []
    for key in sorted(adpts):
        out.extend(adpts[key])
    return out

def sft_lora(m, adpts, scale, W_base, steps=1000, lr=1e-3, seed=0, log_every=250):
    E_spec = m.E[V0:]          # 视图:优化器直接更新特殊 token 嵌入
    h_spec = m.head[:, V0:]
    trainables = flat_adapters(adpts) + [E_spec, h_spec]
    o = MultiAdamW(trainables, lr=lr)
    r = np.random.default_rng(seed)
    for step in range(steps):
        apply_lora(m, adpts, scale, W_base)
        gsum, tot = None, 0.0
        idx = r.integers(0, len(ALL_TASKS), 16)
        for j in idx:
            ids_, mk_ = ALL_TASKS[j]
            loss, grads = loss_and_grads(m, ids_[:-1], ids_[1:], mk_[1:])
            tot += loss
            gs = lora_grads(grads, adpts, scale) + [grads['E'][V0:], grads['head'][:, V0:]]
            if gsum is None:
                gsum = [g / len(idx) for g in gs]
            else:
                for a, g in zip(gsum, gs):
                    a += g / len(idx)
        clip_norms(gsum, 1.0)
        o.step(gsum)
        if (step + 1) % log_every == 0:
            print(f"    step {step+1}: loss {tot/len(idx):.4f}")

def greedy_answer(m, prompt_ids, max_new=5):
    ids = list(prompt_ids)
    for _ in range(max_new):
        logits, _ = m.forward(ids)
        nxt = int(np.argmax(logits[-1]))
        if nxt == T_END:
            break
        ids = ids + [nxt]
    return ids

def task_acc(m):
    ok = 0; total = 0
    for q, a in [(f"反义词:{x}", y) for x, y in ANTI] + [(f"重复:{s}", s) for s in COPY_STRINGS]:
        prompt = [T_USER] + enc(q) + [T_ASSI]
        got = greedy_answer(m, prompt, len(a) + 1)[len(prompt):]
        ok += ("".join(itos.get(i, '#') for i in got) == a); total += 1
    return ok, total

In [ ]:
# ===== 验收:梯度投影 + B=0 不变性(讲义 §4) =====
import copy as _copy
rng0 = np.random.default_rng(0)
m0 = MiniGPT(vocab=8, d=16, n_heads=2, n_layers=1, seed=3)
m0.E = np.vstack([m0.E, rng0.normal(size=(3, 16)) * 0.1])
m0.head = np.hstack([m0.head, rng0.normal(size=(16, 3)) * 0.1])
ad0, Wb0, sc0 = init_lora(m0, keys=LORA_ATTN, r=2, alpha=4)
ids = [3, 4, 8, 5, 9, 10, 7]; tg = ids[1:]; mk = [0, 1, 0, 0, 1, 1]
for key in ad0:                      # 给 B 一点非零值,验收才是非平凡解
    A, B = ad0[key]
    B[:] = np.random.default_rng(2).normal(size=B.shape) * 0.01
apply_lora(m0, ad0, sc0, Wb0)
_, g0 = loss_and_grads(m0, ids[:-1], tg, mk)
ag = lora_grads(g0, ad0, sc0)
flat_ad = flat_adapters(ad0)
gmap = {id(a): g for a, g in zip(flat_ad, ag)}
def f():
    apply_lora(m0, ad0, sc0, Wb0)    # 每次扰动后干净重建
    return loss_and_grads(m0, ids[:-1], tg, mk)[0]
errs = []
rr = np.random.default_rng(1)
for arr in flat_ad:
    for _ in range(6):
        idx = tuple(rr.integers(0, s, 1)[0] for s in arr.shape)
        old = arr[idx]
        arr[idx] = old + 1e-6; fp = f()
        arr[idx] = old - 1e-6; fm = f()
        arr[idx] = old
        errs.append(abs(gmap[id(arr)][idx] - (fp - fm) / 2e-6))
print(f"LoRA 梯度投影抽查 {len(errs)} 坐标 max|err| = {max(errs):.2e}")

gpt_chk = MiniGPT(vocab=8, d=16, n_heads=2, n_layers=1, seed=5)
ad_chk, Wb_chk, sc_chk = init_lora(gpt_chk, keys=LORA_ALL, r=4)
l0, _ = gpt_chk.forward([1, 2, 3])
apply_lora(gpt_chk, ad_chk, sc_chk, Wb_chk)
l1, _ = gpt_chk.forward([1, 2, 3])
print(f"B=0 时挂上 LoRA 前后 logits max|diff| = {np.abs(l0-l1).max():.1e}")

In [ ]:
# ===== 基座(24 课确定性复刻,约 2.2 分钟) =====
t0 = time.time()
gpt = MiniGPT(vocab=V0, d=64, n_heads=4, n_layers=2, seed=0)
params = flat_params(gpt)
opt = MultiAdamW(params)
rng = np.random.default_rng(42)
T, B = 64, 16
for step in range(2000):
    opt.lr = lr_at(step, 2000)
    off = rng.integers(0, len(train_data) - T - 1, B)
    gsum = None
    for b in range(B):
        xb = train_data[off[b]:off[b] + T + 1]
        loss, grads = loss_and_grads(gpt, list(xb[:-1]), list(xb[1:]))
        gs = flat_grads(grads)
        if gsum is None:
            gsum = [g / B for g in gs]
        else:
            for a, g in zip(gsum, gs):
                a += g / B
    clip_norms(gsum, 1.0)
    opt.step(gsum)
extend_vocab(gpt)
bvl, bvl_no = val_loss_no_special(gpt)
print(f"[基座] 全词表 val = {bvl:.4f} | 去特殊 token = {bvl_no:.4f}({time.time()-t0:.0f}s)")
gpt_pure = _copy.deepcopy(gpt)

adpts, W_base, scale = init_lora(gpt, keys=LORA_ALL, r=4, alpha=8)
n_lora = sum(A.size + B.size for A, B in adpts.values())
n_spec = 3 * gpt.D + gpt.D * 3
n_total = 713_600 + 384
print(f"[参数账] adapter {n_lora:,} + 特殊 token {n_spec} = {n_lora+n_spec:,},"
      f"占 {n_total:,} 的 {(n_lora+n_spec)/n_total:.2%}")

In [ ]:
# ===== 主实验:LoRA-SFT + 遗忘分解 + LoRA 回放 + merge(讲义 §5–6) =====
print("[LoRA-SFT] 全线性 r=4 α=8 lr=1e-3 1000 步:")
t1 = time.time()
sft_lora(gpt, adpts, scale, W_base)
acc = task_acc(gpt)
f1, n1 = val_loss_no_special(gpt)
print(f"  任务准确率 {acc[0]}/{acc[1]} | 全词表 val {bvl:.4f} → {f1:.4f}"
      f" | 去特殊 token {n1:.4f} | 用时 {time.time()-t1:.0f}s")

print("\n[LoRA+回放] 半指令半西游记,lr=1e-3,1000 步:")
m3 = _copy.deepcopy(gpt_pure)
ad3, Wb3, sc3 = init_lora(m3, keys=LORA_ALL, r=4, alpha=8)
E_spec = m3.E[V0:]; h_spec = m3.head[:, V0:]
trainables = flat_adapters(ad3) + [E_spec, h_spec]
o3 = MultiAdamW(trainables, lr=1e-3)
rr3 = np.random.default_rng(0)
rp3 = np.random.default_rng(1)
for step in range(1000):
    apply_lora(m3, ad3, sc3, Wb3)
    gsum = None
    idx = rr3.integers(0, len(ALL_TASKS), 8)
    items = [(ALL_TASKS[j][0], ALL_TASKS[j][1]) for j in idx]
    off = rp3.integers(0, len(train_data) - T - 1, 8)
    for o_ in off:
        xb = train_data[o_:o_ + T + 1]
        items.append((list(xb[:-1]), None, list(xb[1:])))
    for item in items:
        if len(item) == 2:
            ids_, mk_ = item
            loss, grads = loss_and_grads(m3, ids_[:-1], ids_[1:], mk_[1:])
        else:
            ids_, _, tg_ = item
            loss, grads = loss_and_grads(m3, ids_, tg_)
        gs = lora_grads(grads, ad3, sc3) + [grads['E'][V0:], grads['head'][:, V0:]]
        if gsum is None:
            gsum = [g / len(items) for g in gs]
        else:
            for a, g in zip(gsum, gs):
                a += g / len(items)
    clip_norms(gsum, 1.0)
    o3.step(gsum)
f3, n3 = val_loss_no_special(m3)
print(f"  任务准确率 {task_acc(m3)[0]}/50 | 全词表 val {bvl:.4f} → {f3:.4f} | 去特殊 token {n3:.4f}")

# merge:把 ΔW 抄进 W,前向逐位不变
l_lora, _ = gpt.forward([1, 2, 3])
for i in range(gpt.NL):
    for k in LORA_ALL:
        gpt.layers[i][k] = gpt.layers[i][k].copy()   # layers 已是 W_base+Δ;弃 adapter
l_merged, _ = gpt.forward([1, 2, 3])
print(f"\n[merge] 合并后 vs 带 adapter 前向 logits max|diff| = {np.abs(l_lora-l_merged).max():.1e}")
for q in ["反义词:大", "重复:筋斗"]:
    prompt = [T_USER] + enc(q) + [T_ASSI]
    got = greedy_answer(gpt, prompt, 5)[len(prompt):]
    print(f"[贪心] {q} → " + repr("".join(itos.get(i, '<|end|>') for i in got)))
out = gpt.generate([stoi[c] for c in "那猴王"], 30, T=0.8, top_p=0.95,
                   rng=np.random.default_rng(1))
print("[续写] " + "".join(itos.get(i, '#') for i in out))

In [ ]:
# ===== 挂对地方 + 秩消融(同一纯底座,600 步;讲义 §7) =====
print("对照(同一纯底座,600 步):")
for name, keys, r_try in [("仅attention r=4", LORA_ATTN, 4),
                          ("全线性 r=1", LORA_ALL, 1),
                          ("全线性 r=4", LORA_ALL, 4),
                          ("全线性 r=16", LORA_ALL, 16)]:
    m2 = _copy.deepcopy(gpt_pure)
    ad2, Wb2, sc2 = init_lora(m2, keys=keys, r=r_try, alpha=8)
    sft_lora(m2, ad2, sc2, Wb2, steps=600, log_every=10**9)
    acc2 = task_acc(m2)
    n_tr = sum(A.size + B.size for A, B in ad2.values()) + 3 * m2.D + m2.D * 3
    print(f"  {name}: acc {acc2[0]}/{acc2[1]} | 可训练 {n_tr:,}")

In [ ]:
# ---- 作业现场 ----

# TODO(C1): 关讲义默写 init_lora 与 apply_lora(干净重建版)

# TODO(C2): 默写 lora_grads 两行投影;数值裁判验收(阈值 1e-8)

# TODO(C3): 手撕 merge:W_merged = W_base + (α/r)B@A,逐位对照前向

# TODO(D1): 7B d=4096 L=32,QLoRA r=16 全线性:adapter 参数量与显存估算

# TODO(E1): α∈{2,8,32} 扫描(固定 r=4):收敛速度与遗忘量

# TODO(E2/预习): 偏好对 (chosen, rejected):如果你只能用一个标量来度量
#                「模型更爱哪个答案」,你用哪个式子?(提示:log 概率比)
